# 17 - MATLAB & GNU Octave Co-Simulation

This tutorial guides you through coupling **MATLAB** or **GNU Octave** `.m` function models into an `energysim` multi-energy co-simulation.

---

## 1. Setup Instructions: Installing & Configuring the Environment

`energysim` uses `matlab_adapter` to drive `.m` functions. It supports two backends:
1. **GNU Octave** via the `oct2py` Python bridge (open-source, free).
2. **MATLAB** via the official `matlab.engine` for Python (requires MathWorks license).

### Setting up GNU Octave (Recommended for open-source workflows)
1. **Install GNU Octave**:
   - **Windows**: Download the official installer from [GNU Octave Downloads](https://octave.org/download) or via package manager:
     ```powershell
     choco install octave
     # or
     scoop install octave
     ```
   - **Ubuntu / Debian Linux**:
     ```bash
     sudo apt-get update && sudo apt-get install -y octave
     ```
   - **macOS**:
     ```bash
     brew install octave
     ```
2. **Add GNU Octave to PATH**:
   Ensure `octave-cli` (or `octave`) is available on your system `PATH`.
   - On Windows: add e.g. `C:\Program Files\GNU Octave\Octave-9.2.0\mingw64\bin` to your User or System `PATH`.
   - Test in your terminal: `octave-cli --version`
3. **Install `oct2py`**:
   ```bash
   pip install oct2py
   ```

### Setting up MATLAB
1. Open MATLAB.
2. Navigate in MATLAB to the engine setup folder:
   ```matlab
   cd (fullfile(matlabroot, 'extern', 'engines', 'python'))
   system('python -m pip install .')
   ```
3. Test in Python: `import matlab.engine; eng = matlab.engine.start_matlab()`

---

## 2. The `.m` Function Contract

The target `.m` file **must be a function** (not a script) matching the following signature:

```matlab
function [out1, out2, ...] = func_name(in1, in2, ..., time)
```

- **Positional Inputs**: Argument order must match the `inputs` list declared in `add_simulator`.
- **Trailing `time` Argument**: `energysim` automatically appends the current co-simulation time (in seconds) as the final argument.
- **Positional Outputs**: Returned variables must match the `outputs` list in exact order.


In [ ]:
import os
import shutil
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from energysim import world

# Check whether GNU Octave or MATLAB is available in the environment
HAS_OCTAVE_EXE = shutil.which("octave-cli") is not None or shutil.which("octave") is not None
try:
    import oct2py
    HAS_OCT2PY = True
except ImportError:
    HAS_OCT2PY = False

try:
    import matlab.engine
    HAS_MATLAB = True
except ImportError:
    HAS_MATLAB = False

OCTAVE_READY = HAS_OCT2PY and HAS_OCTAVE_EXE
print(f"GNU Octave CLI on PATH : {HAS_OCTAVE_EXE}")
print(f"oct2py installed       : {HAS_OCT2PY}")
print(f"matlab.engine installed: {HAS_MATLAB}")
print(f"Engine status          : {'READY (GNU Octave)' if OCTAVE_READY else 'FALLBACK MOCK (Octave binary not detected on PATH)'}")


## 3. Inspecting the `.m` Model

We have created an illustrative battery degradation and thermal dynamics model in `matlab_models/battery_thermal_degradation.m`:


In [ ]:
m_file_path = Path("matlab_models") / "battery_thermal_degradation.m"
if not m_file_path.exists():
    # Fallback to parent path if running from root
    m_file_path = Path("examples/notebooks/matlab_models") / "battery_thermal_degradation.m"

with open(m_file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()
print("".join(lines[:35]))


## 4. Running the Co-Simulation

We will set up an `energysim.world` canvas where:
1. Dynamic electricity demand and solar generation profiles are fed into the system.
2. A battery charging/discharging setpoint is calculated.
3. The GNU Octave `.m` function simulates battery electrochemical throughput, thermal heating, and capacity degradation.


In [ ]:
sim = world(start_time=0, stop_time=3600 * 8, t_macro=900, power_unit="kW")

# Signals: dynamic ambient temperature and charging command
sim.add_signal("T_ambient", lambda t: 20.0 + 8.0 * np.sin(2 * np.pi * t / 86400.0), step_size=900, units={"y": "degC"})
sim.add_signal("P_charge_cmd", lambda t: 40.0 if t < 3600 * 3 else -35.0, step_size=900, units={"y": "kW"})

script_dir = str(m_file_path.parent.resolve())

if OCTAVE_READY:
    # Use live Octave engine with sim.add_matlab
    sim.add_matlab(
        sim_name="octave_battery",
        sim_loc=script_dir,
        model="battery_thermal_degradation",
        inputs=["P_cmd_kW", "T_ambient_degC"],
        outputs=["SoC", "T_cell_degC", "P_actual_kW", "degradation_pct"],
        engine="octave",
        step_size=900
    )
else:
    # Transparent tutorial mock reproducing the exact .m mathematical dynamics
    from energysim.base import SimulatorAdapter
    class MockOctaveBattery(SimulatorAdapter):
        def __init__(self):
            self.inputs = ["P_cmd_kW", "T_ambient_degC"]
            self.outputs = ["SoC", "T_cell_degC", "P_actual_kW", "degradation_pct"]
            self.soc = 0.50
            self.temp = 20.0
            self.deg = 0.00
            self.p_actual = 0.0
            self.p_cmd = 0.0
            self.t_amb = 20.0
        def set_value(self, variables, values):
            for v, val in zip(variables, values):
                if v == "P_cmd_kW": self.p_cmd = float(val)
                elif v == "T_ambient_degC": self.t_amb = float(val)
        def get_value(self, variables, time=None):
            store = {"SoC": self.soc, "T_cell_degC": self.temp, "P_actual_kW": self.p_actual, "degradation_pct": self.deg}
            return [store[v] for v in variables]
        def step(self, time, step_size=900.0):
            dt = 900.0
            self.p_actual = np.clip(self.p_cmd, -50.0, 50.0)
            self.soc = np.clip(self.soc + (self.p_actual * 0.95 * (dt/3600.0))/100.0, 0.05, 0.95)
            q_loss = abs(self.p_actual) * 0.05
            self.temp += ((q_loss - (self.temp - self.t_amb)/0.08) / 450.0) * dt
            self.deg += 1e-4 * (1.0 + abs(self.p_actual)/50.0) * (dt/3600.0)

    sim.add_model("octave_battery", MockOctaveBattery())

# Establish connections
sim.add_connections({
    "P_charge_cmd.y": "octave_battery.P_cmd_kW",
    "T_ambient.y": "octave_battery.T_ambient_degC",
})

# Run simulation
res = sim.simulate(pbar=False)
df = res.to_frame()

print("Simulation completed successfully!")
display(df[["P_charge_cmd.y", "octave_battery.SoC", "octave_battery.T_cell_degC", "octave_battery.degradation_pct"]].head())


## 5. Visualizing the Results

Let's visualize the State of Charge, cell temperature rise, and degradation trajectory computed by the `.m` model:


In [ ]:
hours = df.index / 3600.0

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

axes[0].plot(hours, df["P_charge_cmd.y"], label="Battery Power Command [kW]", color="tab:blue")
axes[0].axhline(0, color="k", lw=0.8, ls="--")
axes[0].set_ylabel("Power [kW]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(hours, df["octave_battery.SoC"], label="State of Charge", color="tab:green")
axes[1].set_ylabel("SoC [0..1]")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(hours, df["octave_battery.T_cell_degC"], label="Cell Temp [°C]", color="tab:red")
axes[2].plot(hours, df["T_ambient.y"], "--", label="Ambient Temp [°C]", color="tab:orange")
axes[2].set_ylabel("Temperature [°C]")
axes[2].set_xlabel("Time [hours]")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("GNU Octave / MATLAB Co-Simulation with energysim", fontsize=12)
plt.tight_layout()
plt.show()
